In [1]:
# Imports

import random

import gymnasium as gym
import numpy as np
import torch

from ppo import (
    Actor,
    Critic,
    collect_rollout,
    compute_gae,
    normalise_advantages,
    compute_ppo_losses
)


In [2]:
# Seed

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed) #before constructing the networks because their initial weights are random


In [3]:
# Environment

env_id = "CartPole-v1"

env = gym.make(env_id)

observation_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("Observation size:", observation_size)
print("Action size:", action_size)

Observation size: 4
Action size: 2


In [4]:
# Actor and critic

hidden_size = 64

actor = Actor(
    observation_size=observation_size,
    action_size=action_size,
    hidden_size=hidden_size
)

critic = Critic(
    observation_size=observation_size,
    hidden_size=hidden_size
)

print(actor)
print()
print(critic)

Actor(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

Critic(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [5]:
# Reset environment

observation, info = env.reset(seed=seed)

env.action_space.seed(seed)

print("Observation:", observation)
print("Observation shape:", observation.shape)

Observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
Observation shape: (4,)


In [6]:
# Convert observation to tensor

observation_tensor = torch.as_tensor(
    observation,
    dtype=torch.float32
)

observation_tensor = observation_tensor.unsqueeze(0)

print("Observation tensor:", observation_tensor)
print("Observation tensor shape:", observation_tensor.shape)

Observation tensor: tensor([[ 0.0274, -0.0061,  0.0359,  0.0197]])
Observation tensor shape: torch.Size([1, 4])


In [7]:
# Get actor logits

with torch.no_grad():

    logits = actor(observation_tensor)

print("Logits:", logits)
print("Logits shape:", logits.shape)

Logits: tensor([[-0.0233, -0.2041]])
Logits shape: torch.Size([1, 2])


In [8]:
# Create action distribution

action_distribution = torch.distributions.Categorical(
    logits=logits
)

print("Action probabilities:", action_distribution.probs)
print("Probability sum:", action_distribution.probs.sum(dim=-1))

Action probabilities: tensor([[0.5451, 0.4549]])
Probability sum: tensor([1.])


In [9]:
# Sample action

with torch.no_grad():

    action_tensor = action_distribution.sample()

print("Action tensor:", action_tensor)
print("Action tensor shape:", action_tensor.shape)

Action tensor: tensor([1])
Action tensor shape: torch.Size([1])


In [10]:
# Calculate old log-probability

with torch.no_grad():

    old_log_prob = action_distribution.log_prob(
        action_tensor
    )

print("Old log-probability:", old_log_prob)
print("Old log-probability shape:", old_log_prob.shape)

Old log-probability: tensor([-0.7876])
Old log-probability shape: torch.Size([1])


In [11]:
# Estimate state value

with torch.no_grad():

    value = critic(observation_tensor)

print("Value:", value)
print("Value shape:", value.shape)

Value: tensor([0.1108])
Value shape: torch.Size([1])


In [12]:
# Check shapes

assert logits.shape == (1, action_size)
assert action_tensor.shape == (1,)
assert old_log_prob.shape == (1,)
assert value.shape == (1,)

print("Actor and critic checks passed.")

# These are worthwhile because they protect us against exactly the kind of silent tensor-shape errors that become painful once PPO is running.

Actor and critic checks passed.


In [13]:
# Take one environment step

action = action_tensor.item()

next_observation, reward, terminated, truncated, info = env.step(
    action
)

print("Action:", action)
print("Next observation:", next_observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)

Action: 1
Next observation: [ 0.02727336  0.18847767  0.03625453 -0.26141977]
Reward: 1.0
Terminated: False
Truncated: False


In [14]:
# Close environment

env.close()

In [15]:
env = gym.make(
    "CartPole-v1",
    max_episode_steps=20
)

observation, info = env.reset(seed=seed)
env.action_space.seed(seed)

num_steps = 256

rollout, observation = collect_rollout(
    env,
    actor,
    critic,
    observation,
    num_steps
)

# Check rollout shapes

assert rollout["states"].shape == (
    num_steps,
    observation_size
)

assert rollout["actions"].shape == (num_steps,)
assert rollout["rewards"].shape == (num_steps,)
assert rollout["old_log_probs"].shape == (num_steps,)
assert rollout["values"].shape == (num_steps,)
assert rollout["next_values"].shape == (num_steps,)
assert rollout["terminateds"].shape == (num_steps,)
assert rollout["truncateds"].shape == (num_steps,)

# Check stored values between consecutive states

episode_ended = torch.logical_or(
    rollout["terminateds"].bool(),
    rollout["truncateds"].bool()
)

for t in range(num_steps - 1):

    if not episode_ended[t]:

        assert torch.allclose(
            rollout["next_values"][t],
            rollout["values"][t + 1]
        )

# NEW: check that episode boundaries actually occurred.
# The alignment check above only tests the boundary case if episodes
# ended inside the rollout.

num_terminations = int(rollout["terminateds"].sum().item())
num_truncations = int(rollout["truncateds"].sum().item())

print("Terminations:", num_terminations)
print("Truncations:", num_truncations)

assert num_terminations > 0
assert num_truncations > 0

# NEW: check the probability ratio is 1 before any update.
# Recomputing log-probabilities from the stored states and actions with the
# same actor must reproduce the stored old log-probabilities.

with torch.no_grad():

    logits = actor(rollout["states"])

    action_distribution = torch.distributions.Categorical(
        logits=logits
    )

    new_log_probs = action_distribution.log_prob(
        rollout["actions"]
    )

log_ratio = new_log_probs - rollout["old_log_probs"]

ratio = torch.exp(log_ratio)

assert ratio.shape == (num_steps,)

assert torch.allclose(
    ratio,
    torch.ones(num_steps)
)

print("Rollout checks passed.")

env.close()

Terminations: 9
Truncations: 7
Rollout checks passed.


In [16]:
# GAE termination test

gamma = 0.99
gae_lambda = 0.95

test_rollout = {
    "rewards": torch.tensor([1.0, 1.0, 1.0]),
    "values": torch.tensor([0.5, 0.6, 0.7]),
    "next_values": torch.tensor([0.6, 0.7, 0.8]),
    "terminateds": torch.tensor([0.0, 0.0, 1.0]),
    "truncateds": torch.tensor([0.0, 0.0, 0.0])
}

test_advantages, test_returns = compute_gae(
    test_rollout,
    gamma,
    gae_lambda
)

delta_2 = 1.0 - 0.7
advantage_2 = delta_2

delta_1 = (
    1.0
    + gamma * 0.7
    - 0.6
)

advantage_1 = (
    delta_1
    + gamma
    * gae_lambda
    * advantage_2
)

delta_0 = (
    1.0
    + gamma * 0.6
    - 0.5
)

advantage_0 = (
    delta_0
    + gamma
    * gae_lambda
    * advantage_1
)

expected_advantages = torch.tensor([
    advantage_0,
    advantage_1,
    advantage_2
])

expected_returns = (
    expected_advantages
    + test_rollout["values"]
)

assert torch.allclose(
    test_advantages,
    expected_advantages
)

assert torch.allclose(
    test_returns,
    expected_returns
)

print("Expected advantages:", expected_advantages)
print("GAE advantages:     ", test_advantages)
print("Termination GAE check passed.")


Expected advantages: tensor([2.3873, 1.3751, 0.3000])
GAE advantages:      tensor([2.3873, 1.3752, 0.3000])
Termination GAE check passed.


In [17]:
# GAE truncation test

test_rollout = {
    "rewards": torch.tensor([1.0, 1.0, 1.0, 1.0]),
    "values": torch.tensor([0.5, 0.6, -0.2, 0.3]),
    "next_values": torch.tensor([0.6, 0.8, 0.3, 0.4]),
    "terminateds": torch.tensor([0.0, 0.0, 0.0, 1.0]),
    "truncateds": torch.tensor([0.0, 1.0, 0.0, 0.0])
}

test_advantages, test_returns = compute_gae(
    test_rollout,
    gamma,
    gae_lambda
)

# The truncated transition keeps the value bootstrap
expected_truncated_delta = (
    1.0
    + gamma * 0.8
    - 0.6
)

# The GAE recursion stops at the truncation boundary
assert torch.allclose(
    test_advantages[1],
    torch.tensor(expected_truncated_delta)
)

print(
    "Expected advantage at truncation:",
    expected_truncated_delta
)

print(
    "GAE advantage at truncation:     ",
    test_advantages[1].item()
)

print("Truncation GAE check passed.")


Expected advantage at truncation: 1.1920000000000002
GAE advantage at truncation:      1.1920000314712524
Truncation GAE check passed.


In [18]:
# GAE on the CartPole rollout

advantages, returns = compute_gae(
    rollout,
    gamma,
    gae_lambda
)

assert advantages.shape == (num_steps,)
assert returns.shape == (num_steps,)
assert torch.isfinite(advantages).all()
assert torch.isfinite(returns).all()

print("Advantage shape:", advantages.shape)
print("Return shape:", returns.shape)
print("CartPole GAE checks passed.")


Advantage shape: torch.Size([256])
Return shape: torch.Size([256])
CartPole GAE checks passed.


In [19]:
# Advantage normalisation

normalised_advantages = normalise_advantages(
    advantages
)

assert normalised_advantages.shape == (num_steps,)
assert torch.isfinite(normalised_advantages).all()
assert abs(normalised_advantages.mean().item()) < 1e-6
assert abs(normalised_advantages.std().item() - 1.0) < 1e-5

print(
    "Mean before normalisation:",
    advantages.mean().item()
)

print(
    "Mean after normalisation:",
    normalised_advantages.mean().item()
)

print(
    "Standard deviation after normalisation:",
    normalised_advantages.std().item()
)

print("Advantage normalisation check passed.")


Mean before normalisation: 6.6187520027160645
Mean after normalisation: 1.1734664440155029e-07
Standard deviation after normalisation: 1.0
Advantage normalisation check passed.


In [20]:
# PPO loss calculations

clip_epsilon = 0.2

ppo_results = compute_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon
)

ratios = ppo_results["ratios"]

assert ratios.shape == (num_steps,)
assert ppo_results["new_log_probs"].shape == (num_steps,)
assert ppo_results["new_values"].shape == (num_steps,)
assert torch.isfinite(ppo_results["policy_loss"])
assert torch.isfinite(ppo_results["value_loss"])
assert torch.isfinite(ppo_results["entropy"])

print("Policy loss:", ppo_results["policy_loss"].item())
print("Value loss: ", ppo_results["value_loss"].item())
print("Entropy:    ", ppo_results["entropy"].item())
print("PPO loss calculations passed.")


Policy loss: -1.0617077350616455e-07
Value loss:  27.039709091186523
Entropy:     0.688571572303772
PPO loss calculations passed.


In [21]:
# PPO ratio test

print("Mean ratio:   ", ratios.mean().item())
print("Minimum ratio:", ratios.min().item())
print("Maximum ratio:", ratios.max().item())

assert torch.allclose(
    ratios,
    torch.ones_like(ratios),
    atol=1e-6
)

print("PPO ratio check passed.")

Mean ratio:    1.0
Minimum ratio: 0.9999998807907104
Maximum ratio: 1.0000001192092896
PPO ratio check passed.


In [22]:
# PPO clipping test

test_states = rollout["states"][:4]
test_actions = rollout["actions"][:4]

test_ratios = torch.tensor([1.3, 0.7, 1.3, 0.7])
test_advantages = torch.tensor([1.0, -1.0, -1.0, 1.0])

with torch.no_grad():

    test_logits = actor(test_states)
    test_distribution = torch.distributions.Categorical(logits=test_logits)
    test_new_log_probs = test_distribution.log_prob(test_actions)

# Choose old log-probabilities so the function produces the target ratios
test_old_log_probs = test_new_log_probs - torch.log(test_ratios)

test_rollout = {
    "states": test_states,
    "actions": test_actions,
    "old_log_probs": test_old_log_probs
}

test_returns = returns[:4]

test_results = compute_ppo_losses(
    actor,
    critic,
    test_rollout,
    test_advantages,
    test_returns,
    clip_epsilon
)

expected_unclipped = torch.tensor([1.3, -0.7, -1.3, 0.7])
expected_clipped = torch.tensor([1.2, -0.8, -1.2, 0.8])
expected_objective = torch.tensor([1.2, -0.8, -1.3, 0.7])

assert torch.allclose(
    test_results["ratios"],
    test_ratios,
    atol=1e-6
)

assert torch.allclose(
    test_results["unclipped_objective"],
    expected_unclipped,
    atol=1e-6
)

assert torch.allclose(
    test_results["clipped_objective"],
    expected_clipped,
    atol=1e-6
)

expected_policy_loss = -expected_objective.mean()

assert torch.allclose(
    test_results["policy_loss"],
    expected_policy_loss,
    atol=1e-6
)

print("Ratios:             ", test_results["ratios"])
print("Unclipped objective:", test_results["unclipped_objective"])
print("Clipped objective:  ", test_results["clipped_objective"])
print("Expected objective: ", expected_objective)
print("PPO clipping function check passed.")

Ratios:              tensor([1.3000, 0.7000, 1.3000, 0.7000], grad_fn=<ExpBackward0>)
Unclipped objective: tensor([ 1.3000, -0.7000, -1.3000,  0.7000], grad_fn=<MulBackward0>)
Clipped objective:   tensor([ 1.2000, -0.8000, -1.2000,  0.8000], grad_fn=<MulBackward0>)
Expected objective:  tensor([ 1.2000, -0.8000, -1.3000,  0.7000])
PPO clipping function check passed.


In [23]:
# Value loss and entropy checks

value_loss = ppo_results["value_loss"]
entropy = ppo_results["entropy"]

assert value_loss.item() >= 0.0
assert entropy.item() >= 0.0
assert entropy.item() <= np.log(action_size) + 1e-6

print("Value loss:", value_loss.item())
print("Entropy:   ", entropy.item())
print("Value loss and entropy checks passed.")


Value loss: 27.039709091186523
Entropy:    0.688571572303772
Value loss and entropy checks passed.


In [ ]:
# One full-batch PPO update

# Fixed optimisation settings for the initial PPO baseline
learning_rate = 3e-4
entropy_coefficient = 0.01
max_grad_norm = 0.5

# Separate optimisers because the actor and critic are separate networks
actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=learning_rate
)

critic_optimizer = torch.optim.Adam(
    critic.parameters(),
    lr=learning_rate
)

# Save copies of the parameters before the update
# detach().clone() gives fixed copies that will not change with the network
actor_parameters_before = []

for parameter in actor.parameters():
    actor_parameters_before.append(
        parameter.detach().clone()
    )

critic_parameters_before = []

for parameter in critic.parameters():
    critic_parameters_before.append(
        parameter.detach().clone()
    )

# Recalculate the PPO losses using the full original rollout
# This is still before any policy update, so the ratios should start near 1
ppo_results = compute_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon
)

# Actor loss:
# policy_loss is already the negative clipped PPO objective
# entropy is subtracted because higher entropy should reduce the loss
actor_loss = (
    ppo_results["policy_loss"]
    - entropy_coefficient * ppo_results["entropy"]
)

# The critic has its own network and optimiser, so its loss is separate
critic_loss = ppo_results["value_loss"]

print("Actor loss before update:", actor_loss.item())
print("Critic loss before update:", critic_loss.item())
print("Entropy before update:", ppo_results["entropy"].item())

# Remove any gradients left over from previous calculations
actor_optimizer.zero_grad()
critic_optimizer.zero_grad()

# Calculate gradients for the actor and critic
actor_loss.backward()
critic_loss.backward()

# Clip the gradient norms before the optimiser uses them
# clip_grad_norm_ returns the total norm before clipping
actor_gradient_norm = torch.nn.utils.clip_grad_norm_(
    actor.parameters(),
    max_grad_norm
)

critic_gradient_norm = torch.nn.utils.clip_grad_norm_(
    critic.parameters(),
    max_grad_norm
)

print("Actor gradient norm before clipping:", actor_gradient_norm.item())
print("Critic gradient norm before clipping:", critic_gradient_norm.item())

# Apply one optimiser step to each network
actor_optimizer.step()
critic_optimizer.step()

print("One actor and critic optimiser step completed.")

Actor loss before update: -0.006885821931064129
Critic loss before update: 27.039709091186523
Entropy before update: 0.688571572303772
Actor gradient norm before clipping: 0.16786907613277435
Critic gradient norm before clipping: 14.079136848449707
One actor and critic optimiser step completed.


In [25]:
# Check the one-step PPO update

# Check whether at least one actor parameter changed
actor_changed = False

for before, after in zip(
    actor_parameters_before,
    actor.parameters()
):

    if not torch.equal(before, after):
        actor_changed = True

# Check whether at least one critic parameter changed
critic_changed = False

for before, after in zip(
    critic_parameters_before,
    critic.parameters()
):

    if not torch.equal(before, after):
        critic_changed = True

assert actor_changed
assert critic_changed

print("Actor parameters changed:", actor_changed)
print("Critic parameters changed:", critic_changed)

# Re-evaluate the SAME rollout using the updated actor
#
# The stored old_log_probs still come from the policy that collected
# the rollout, while the actor parameters have now changed.
post_update_results = compute_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon
)

post_update_ratios = post_update_results["ratios"]

# Before the update, all ratios were approximately 1.
# After changing the actor, at least some should move away from 1.
ratio_changes = torch.abs(
    post_update_ratios - 1.0
)

max_ratio_change = ratio_changes.max().item()

print()
print("Mean ratio after update:", post_update_ratios.mean().item())
print("Minimum ratio after update:", post_update_ratios.min().item())
print("Maximum ratio after update:", post_update_ratios.max().item())
print("Maximum change from 1:", max_ratio_change)

assert max_ratio_change > 1e-6

print()
print("One PPO gradient update check passed.")

Actor parameters changed: True
Critic parameters changed: True

Mean ratio after update: 1.0003430843353271
Minimum ratio after update: 0.9786659479141235
Maximum ratio after update: 1.0277549028396606
Maximum change from 1: 0.027754902839660645

One PPO gradient update check passed.


In [ ]:
# Minibatch and repeated epoch check

update_epochs = 4
minibatch_size = 64

num_steps = rollout["states"].shape[0]

# Check that the rollout divides evenly into minibatches
assert num_steps % minibatch_size == 0

num_minibatches = num_steps // minibatch_size
expected_optimizer_steps = num_minibatches * update_epochs

print("Rollout steps:", num_steps)
print("Minibatch size:", minibatch_size)
print("Minibatches per epoch:", num_minibatches)
print("PPO epochs:", update_epochs)
print("Expected optimiser steps per network:", expected_optimizer_steps)

# Save copies of the quantities that should remain fixed
# while the same rollout is reused for several PPO epochs
old_log_probs_before = rollout["old_log_probs"].clone()
advantages_before = normalised_advantages.clone()
returns_before = returns.clone()

# Reuse the same rollout for several PPO epochs
update_ppo(
    actor,
    critic,
    actor_optimizer,
    critic_optimizer,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon,
    entropy_coefficient,
    update_epochs,
    minibatch_size,
    max_grad_norm
)

# Check that the fixed rollout quantities were not changed
assert torch.equal(
    rollout["old_log_probs"],
    old_log_probs_before
)

assert torch.equal(
    normalised_advantages,
    advantages_before
)

assert torch.equal(
    returns,
    returns_before
)

print()
print("Old log-probabilities stayed fixed.")
print("Advantages stayed fixed.")
print("Return targets stayed fixed.")
print("Repeated PPO update completed.")

In [ ]:
# Re-evaluate the same rollout after repeated PPO epochs

post_epoch_results = compute_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon
)

post_epoch_ratios = post_epoch_results["ratios"]

print(
    "Mean ratio after repeated epochs:",
    post_epoch_ratios.mean().item()
)

print(
    "Minimum ratio after repeated epochs:",
    post_epoch_ratios.min().item()
)

print(
    "Maximum ratio after repeated epochs:",
    post_epoch_ratios.max().item()
)